## [HabitSync — 🇧🇷MVP de Recomendação por Voz para Casais / 🇬🇧Voice Recommendation MVP for Couples]()

<br><br>

### [***Estudo de Caso — 🇧🇷Assistente Inteligente de Recomendação (2026) / 🇬🇧Case Study — Intelligent Recommendation Assistant (2026)***]
<br>

Este notebook evolui diretamente o protótipo `assistenteVoz.ipynb` (anteriormente
prototipado sob o nome de trabalho **"NewVoiceHabits"**, renomeado para
**HabitSync**), implementando o pipeline de recomendação híbrido descrito no
briefing: filtragem baseada em conteúdo, filtragem colaborativa **simulada**,
reconciliação de perfil duplo (casal) e ranking final ponderado.

---

*This notebook directly evolves the `assistenteVoz.ipynb` prototype (previously
prototyped under the working name **"NewVoiceHabits"**, renamed to
**HabitSync**), implementing the hybrid recommendation pipeline described in
the briefing: content-based filtering, **simulated** collaborative filtering,
double-profile (couple) reconciliation, and weighted final ranking.*

<br>

### [***🇧🇷Avisos Importantes / 🇬🇧Important Notices***]
<br>

English: "All data in this notebook is 100% synthetic and generated for
educational purposes only. Recommendations are informational and do not
replace professional nutrition, medical, or training guidance. The
collaborative signal is simulated and does not represent real market
behavior."

Português (Brasil): "Todos os dados deste notebook são 100% sintéticos,
gerados apenas para fins educacionais. As recomendações são informativas e
não substituem orientação profissional de nutrição, saúde ou treinamento.
O sinal colaborativo é simulado e não representa comportamento real de
mercado."

In [70]:
# Instalação (ambiente Google Colab)
!pip install gTTS -q


In [71]:
import pandas as pd
import numpy as np
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


## [1. 🇧🇷Geração do Dataset Sintético / 🇬🇧Synthetic Dataset Generation]()

### ***🇧🇷Produtos, casais, usuários e interações / 🇬🇧Products, couples, users, and interactions***

🇧🇷
Reproduz exatamente o dataset validado na Etapa 1 (mesma seed, mesma lógica).
Todos os registros carregam `is_synthetic = True`. O sinal colaborativo
(`interactions`) é inteiramente simulado, para fins educacionais.

#

🇬🇧
*Reproduces exactly the dataset validated in Step 1 (same seed, same logic).*
*All records carry `is_synthetic = True`. The collaborative signal (`interactions`) is entirely simulated, for educational purposes.*

In [72]:
CATEGORIES = {
    "protein": ["whey", "isolate", "hydrolyzed", "vegan_protein"],
    "creatine": ["creatine_monohydrate", "creatine_hcl"],
    "vitamins": ["multivitamin", "vitamin_d", "omega_3"],
    "healthy_food": ["snack_bar", "granola", "oatmeal"],
    "equipment": ["resistance_bands", "yoga_mat", "dumbbell_set"],
}
BRANDS = ["VitaFit", "PureNutri", "GreenGain", "IronCore", "BalancePlus"]
CURRENCY = "BRL"

GOALS = ["muscle_gain", "weight_loss", "endurance"]
RESTRICTIONS = ["vegan", "vegetarian", "lactose_free", "gluten_free", "nut_allergy", "none"]
TRAINING_TYPES = ["bodybuilding", "crossfit", "yoga"]


def _allergens_for(category, subcategory):
    """Heurística didática de alérgenos por subcategoria (sem alegação clínica)."""
    if category == "equipment":
        return []
    if subcategory == "vegan_protein":
        pool = ["soy", "gluten"]
        return random.sample(pool, k=random.randint(0, 1))
    if subcategory in ("whey", "hydrolyzed"):
        allergens = ["lactose"] if random.random() < 0.8 else []
        if random.random() < 0.15:
            allergens.append("soy")
        return allergens
    if subcategory == "isolate":
        return ["lactose"] if random.random() < 0.3 else []
    if category == "creatine":
        return ["soy"] if random.random() < 0.1 else []
    if category == "vitamins":
        return random.sample(["gluten", "soy"], k=random.randint(0, 1))
    if category == "healthy_food":
        pool = ["gluten", "nuts"]
        return random.sample(pool, k=random.randint(0, 2))
    return []


def _tags_for(category, subcategory, allergens):
    tags = set()
    if category != "equipment":
        if ("lactose" not in allergens and "egg" not in allergens
                and (subcategory == "vegan_protein" or random.random() < 0.25)):
            tags.add("#vegan")
        tags.add("#vegetarian")
        if "gluten" not in allergens:
            tags.add("#gluten_free")
        if "lactose" not in allergens:
            tags.add("#lactose_free")
        if random.random() < 0.2:
            tags.add("#low_sugar")
    return sorted(tags)


In [73]:
def make_products(n=25):
    rows = []
    for i in range(1, n + 1):
        category = random.choice(list(CATEGORIES.keys()))
        subcategory = random.choice(CATEGORIES[category])
        allergens = _allergens_for(category, subcategory)
        tags = _tags_for(category, subcategory, allergens)
        is_food = category != "equipment"
        rows.append({
            "product_id": f"P{i:03d}",
            "category": category,
            "subcategory": subcategory,
            "brand": random.choice(BRANDS),
            "price": round(random.uniform(19.9, 249.9), 2),
            "currency": CURRENCY,
            "ingredients": f"generic {subcategory.replace('_', ' ')} formula" if is_food else "N/A (non-food item)",
            "allergens": ",".join(allergens) if allergens else "none",
            "description": f"{subcategory.replace('_', ' ').title()} product for {category.replace('_', ' ')} goals",
            "tags": ",".join(tags) if tags else "none",
            "availability": bool(random.random() < 0.9),
            "is_synthetic": True,
        })

    # Produtos curados: garantem cobertura determinística dos cenários de demonstração
    # (nomes/marcas fictícios e genéricos, sem alegação de eficácia clínica)
    curated = [
        {"product_id": "P901", "category": "protein", "subcategory": "vegan_protein",
         "brand": "GreenGain", "price": 129.9, "currency": CURRENCY,
         "ingredients": "generic pea and rice protein blend", "allergens": "none",
         "description": "Vegan Protein Blend for muscle gain goals",
         "tags": "#vegan,#vegetarian,#gluten_free,#lactose_free", "availability": True,
         "is_synthetic": True},
        {"product_id": "P902", "category": "healthy_food", "subcategory": "snack_bar",
         "brand": "IronCore", "price": 12.5, "currency": CURRENCY,
         "ingredients": "generic protein bar with mixed nuts", "allergens": "nuts,gluten",
         "description": "High-Protein Nut Bar for muscle gain goals",
         "tags": "#vegetarian", "availability": True,
         "is_synthetic": True},
        {"product_id": "P903", "category": "protein", "subcategory": "isolate",
         "brand": "PureNutri", "price": 139.9, "currency": CURRENCY,
         "ingredients": "generic whey isolate formula", "allergens": "none",
         "description": "Nut-Free Whey Isolate for muscle gain goals",
         "tags": "#vegetarian,#gluten_free,#lactose_free", "availability": True,
         "is_synthetic": True},
    ]
    return pd.concat([pd.DataFrame(rows), pd.DataFrame(curated)], ignore_index=True)


def make_users(n_couples=10):
    rows = []
    for c in range(1, n_couples + 1):
        couple_id = f"C{c:03d}"
        for role in ["partner_a", "partner_b"]:
            rows.append({
                "user_id": f"{couple_id}_{role}", "couple_id": couple_id, "role": role,
                "goals": random.choice(GOALS),
                "dietary_restrictions": random.choice(RESTRICTIONS),
                "training_type": random.choice(TRAINING_TYPES),
                "is_synthetic": True,
            })
    return pd.DataFrame(rows)


def add_demo_couples(users_df):
    """Casais curados (determinísticos) usados nos cenários 3 e 4 da demonstração."""
    demo_rows = [
        {"user_id": "DEMO_COMPAT_partner_a", "couple_id": "DEMO_COMPAT", "role": "partner_a",
         "goals": "muscle_gain", "dietary_restrictions": "vegan", "training_type": "bodybuilding",
         "is_synthetic": True},
        {"user_id": "DEMO_COMPAT_partner_b", "couple_id": "DEMO_COMPAT", "role": "partner_b",
         "goals": "muscle_gain", "dietary_restrictions": "vegetarian", "training_type": "crossfit",
         "is_synthetic": True},
        {"user_id": "DEMO_CONFLICT_partner_a", "couple_id": "DEMO_CONFLICT", "role": "partner_a",
         "goals": "muscle_gain", "dietary_restrictions": "none", "training_type": "bodybuilding",
         "is_synthetic": True},
        {"user_id": "DEMO_CONFLICT_partner_b", "couple_id": "DEMO_CONFLICT", "role": "partner_b",
         "goals": "endurance", "dietary_restrictions": "nut_allergy", "training_type": "yoga",
         "is_synthetic": True},
    ]
    return pd.concat([users_df, pd.DataFrame(demo_rows)], ignore_index=True)


In [74]:
def _compat_weight(product, user):
    score = 0
    if user["dietary_restrictions"] == "vegan" and "#vegan" in str(product["tags"]):
        score += 2
    if user["goals"] == "muscle_gain" and product["category"] in ("protein", "creatine"):
        score += 2
    if user["goals"] == "weight_loss" and product["category"] in ("healthy_food", "vitamins"):
        score += 1
    if user["training_type"] == "yoga" and product["category"] == "equipment":
        score += 1
    return score + 1  # peso mínimo, permite exploração


def make_interactions(users_df, products_df, per_user_min=5, per_user_max=10):
    import datetime as _dt
    rows = []
    interaction_id = 1
    base_date = _dt.datetime(2026, 1, 1)
    event_types = ["view", "rating", "purchase"]

    for _, user in users_df.iterrows():
        weights = products_df.apply(lambda p: _compat_weight(p, user), axis=1).values.astype(float)
        weights = weights / weights.sum()
        n_events = random.randint(per_user_min, per_user_max)
        chosen = np.random.choice(products_df["product_id"], size=n_events, p=weights, replace=True)
        for product_id in chosen:
            event_type = random.choices(event_types, weights=[0.5, 0.3, 0.2])[0]
            value = round(random.uniform(3.0, 5.0), 1) if event_type == "rating" else 1
            rows.append({
                "interaction_id": f"I{interaction_id:04d}", "user_id": user["user_id"],
                "product_id": product_id, "event_type": event_type, "value": value,
                "timestamp": (base_date + _dt.timedelta(days=random.randint(0, 120))).isoformat(),
                "is_synthetic": True,
            })
            interaction_id += 1
    return pd.DataFrame(rows)


def validate_dataset(products_df, users_df, interactions_df):
    """Validações obrigatórias (assert-based) antes de liberar o dataset."""
    assert products_df["is_synthetic"].all(), "Existem produtos não marcados como sintéticos"
    assert users_df["is_synthetic"].all(), "Existem usuários não marcados como sintéticos"
    assert interactions_df["is_synthetic"].all(), "Existem interações não marcadas como sintéticas"

    counts = users_df.groupby("couple_id")["user_id"].count()
    assert (counts == 2).all(), f"Casais com número de usuários != 2: {counts[counts != 2].to_dict()}"

    required_cols = ["category", "subcategory", "price", "currency", "ingredients",
                      "allergens", "tags", "availability"]
    for col in required_cols:
        assert col in products_df.columns, f"Coluna obrigatória ausente em products: {col}"
        assert products_df[col].notna().all(), f"Valores nulos encontrados em products.{col}"

    assert interactions_df["user_id"].isin(users_df["user_id"]).all(), "interaction com user_id inválido"
    assert interactions_df["product_id"].isin(products_df["product_id"]).all(), "interaction com product_id inválido"

    assert "P902" in products_df["product_id"].values, "Produto de conflito (nuts) ausente"
    assert "P903" in products_df["product_id"].values, "Produto seguro alternativo ausente"

    print("[OK] validate_dataset: todas as verificações passaram.")


In [75]:
products_df = make_products(n=25)
users_df = add_demo_couples(make_users(n_couples=10))
interactions_df = make_interactions(users_df, products_df)

validate_dataset(products_df, users_df, interactions_df)
print(f"products: {products_df.shape} | users: {users_df.shape} | interactions: {interactions_df.shape}")
print(f"couples: {users_df['couple_id'].nunique()} (incluindo 2 casais de demonstração)")
products_df.head(3)

[OK] validate_dataset: todas as verificações passaram.
products: (28, 12) | users: (24, 7) | interactions: (180, 7)
couples: 12 (incluindo 2 casais de demonstração)


,product_id,category,subcategory,brand,price,currency,ingredients,allergens,description,tags,availability,is_synthetic
0,P001,protein,whey,VitaFit,175.54,BRL,generic whey formula,lactose,Whey product for protein goals,"#gluten_free,#low_sugar,#vegetarian",True,True
1,P002,protein,vegan_protein,PureNutri,73.41,BRL,generic vegan protein formula,none,Vegan Protein product for protein goals,"#gluten_free,#lactose_free,#low_sugar,#vegan,#...",True,True
2,P003,equipment,resistance_bands,BalancePlus,116.39,BRL,N/A (non-food item),none,Resistance Bands product for equipment goals,none,True,True


## [2. 🇧🇷Internacionalização  / 🇬🇧Internationalization ]()
<br><br>

### [***🇧🇷Dicionário de strings de interface / 🇬🇧Interface strings dictionary***]

<br>

🇧🇷
Todas as mensagens de sistema (saudação, erro, encerramento e o aviso de
dados sintéticos) existem em português e inglês.

#
🇬🇧
*All system messages (greeting, error, closing, and the synthetic data warning) exist in both Portuguese and English.*

In [76]:
STRINGS = {
    "banner_synthetic": {
        "pt": "Aviso: todos os dados usados sao sinteticos (fins educacionais). "
              "As recomendacoes sao informativas e nao substituem orientacao "
              "nutricional, medica ou de treinamento profissional.",
        "en": "Notice: all data used is synthetic (educational purposes only). "
              "Recommendations are informational and do not replace professional "
              "nutrition, medical, or training guidance.",
    },
    "greet": {"pt": "Ola! Posso recomendar produtos de treino e dieta para voces.",
              "en": "Hi! I can recommend training and diet products for you two."},
    "unknown": {"pt": "Nao entendi o comando. Tente pedir uma recomendacao.",
                "en": "I did not understand. Try asking for a recommendation."},
    "exit": {"pt": "Ate logo!", "en": "Goodbye!"},
    "command_dropdown_desc": {
        "pt": "Comando Pré-definido:",
        "en": "Pre-defined Command:",
    },
    "custom_command_desc": {
        "pt": "Comando Personalizado:",
        "en": "Custom Command:",
    },
    "custom_command_placeholder": {
        "pt": "Digite seu comando personalizado aqui...",
        "en": "Enter your custom command here...",
    },
    "couple_id_desc": {
        "pt": "Casal:",
        "en": "Couple:",
    },
    "select_couple_placeholder": {
        "pt": "Selecione um casal...",
        "en": "Select a couple...",
    },
    "couple_incomplete_info": {
        "pt": " (Informação incompleta)",
        "en": " (Incomplete information)",
    },
    "partner_a_label": {
        "pt": "P.A:",
        "en": "Partner A:",
    },
    "partner_b_label": {
        "pt": "P.B:",
        "en": "Partner B:",
    },
    "language_dropdown_desc": {
        "pt": "Idioma:",
        "en": "Language:",
    },
    "submit_button_desc": {
        "pt": "Obter Recomendação",
        "en": "Get Recommendation",
    },
    "submit_button_tooltip": {
        "pt": "Clique para obter a recomendação",
        "en": "Click to get the recommendation",
    },
    "tab_suggested_commands_title": {
        "pt": "Comandos Sugeridos",
        "en": "Suggested Commands",
    },
    "tab_custom_command_title": {
        "pt": "Comando Personalizado",
        "en": "Custom Command",
    },
    "error_empty_input": {
        "pt": "Por favor, selecione um comando pré-definido OU digite um comando personalizado E um ID de casal.",
        "en": "Please select a pre-defined command OR enter a custom command AND a couple ID.",
    },
    "processing_command": {
        "pt": "Processando comando: '{user_command}' para o casal '{couple_id}' no idioma '{selected_lang}'...",
        "en": "Processing command: '{user_command}' for couple '{couple_id}' in language '{selected_lang}'...",
    },
    "audio_error": {
        "pt": "Erro ao reproduzir áudio: {e}",
        "en": "Error playing audio: {e}",
    }
}


def t(key, lang="pt"):
    return STRINGS[key].get(lang, STRINGS[key]["pt"])


print(t("banner_synthetic", "pt"))
print(t("banner_synthetic", "en"))

Aviso: todos os dados usados sao sinteticos (fins educacionais). As recomendacoes sao informativas e nao substituem orientacao nutricional, medica ou de treinamento profissional.
Notice: all data used is synthetic (educational purposes only). Recommendations are informational and do not replace professional nutrition, medical, or training guidance.


## [3. 🇧🇷Prioridade Absoluta: Restrições Alimentares e Alérgenos / 🇬🇧Absolute Priority: Dietary Restrictions and Allergens]()
### ***🇧🇷Regra de negócio inegociável / 🇬🇧Non-negotiable business rule***



🇧🇷
Antes de qualquer pontuação por conteúdo ou sinal colaborativo, todo produto
passa por um filtro de segurança. Um produto só entra na lista de
candidatos se for seguro para **os dois parceiros simultaneamente** — isto
tem prioridade sobre objetivos e preferências.

#

🇬🇧
*Before any content or collaborative scoring, every product goes through a safety filter. A product only enters the candidate list if it is safe for **both partners simultaneously** — this takes priority over goals and preferences.*

In [77]:
RESTRICTION_RULE = {
    "vegan": lambda p: "#vegan" in str(p["tags"]),
    "vegetarian": lambda p: "#vegetarian" in str(p["tags"]),
    "lactose_free": lambda p: "lactose" not in str(p["allergens"]),
    "gluten_free": lambda p: "gluten" not in str(p["allergens"]),
    "nut_allergy": lambda p: "nuts" not in str(p["allergens"]),
    "none": lambda p: True,
}


def is_safe_for_user(product, user):
    rule = RESTRICTION_RULE.get(user["dietary_restrictions"], lambda p: True)
    return bool(rule(product))


def is_safe_for_couple(product, user_a, user_b):
    return is_safe_for_user(product, user_a) and is_safe_for_user(product, user_b)


## [4. 🇧🇷Filtragem Baseada em Conteúdo / 🇬🇧Content-Based Filtering]()

### ***🇧🇷Pontuação por objetivo e tipo de treino / 🇬🇧Scoring by goal and training type***


🇧🇷
Aplicada apenas sobre os candidatos já considerados seguros (Seção 3).

#

🇬🇧
*Applied only on candidates already considered safe (Section 3).*

In [78]:
GOAL_CATEGORY = {
    "muscle_gain": ["protein", "creatine"],
    "weight_loss": ["healthy_food", "vitamins"],
    "endurance": ["vitamins", "healthy_food"],
}
TRAINING_CATEGORY_BONUS = {
    "yoga": ["equipment"],
    "crossfit": ["equipment", "protein"],
    "bodybuilding": ["protein", "creatine"],
}


def content_score_for_user(product, user):
    score = 0.0
    if product["category"] in GOAL_CATEGORY.get(user["goals"], []):
        score += 2.0
    if product["category"] in TRAINING_CATEGORY_BONUS.get(user["training_type"], []):
        score += 1.0
    return score


def content_scores_for_couple(candidates_df, user_a, user_b):
    score_a = candidates_df.apply(lambda p: content_score_for_user(p, user_a), axis=1)
    score_b = candidates_df.apply(lambda p: content_score_for_user(p, user_b), axis=1)
    return (score_a + score_b) / 2.0


## [5. 🇧🇷Filtragem Colaborativa (Simulada) / 🇬🇧Collaborative Filtering (Simulated)]()
<br><br>

### [***🇧🇷Similaridade entre usuários sintéticos / 🇬🇧Similarity between synthetic users***]

<br>

🇧🇷
Calculada por similaridade de cosseno sobre a matriz usuário×produto de
`interactions.csv`. **Este sinal é inteiramente simulado e educacional —
não representa comportamento real de mercado.**

#

🇬🇧
*Calculated via cosine similarity on the user×product matrix from `interactions.csv`. **This signal is entirely simulated and educational — it does not represent real market behavior.***

In [79]:
def build_interaction_matrix(interactions_df, products_df):
    matrix = pd.pivot_table(
        interactions_df, index="user_id", columns="product_id",
        values="value", aggfunc="sum", fill_value=0.0,
    )
    matrix = matrix.reindex(columns=products_df["product_id"], fill_value=0.0)
    return matrix


def simulate_collaborative_filter(user_id, interaction_matrix):
    if user_id not in interaction_matrix.index:
        return pd.Series(0.0, index=interaction_matrix.columns)
    target = interaction_matrix.loc[user_id].values
    others = interaction_matrix.drop(index=user_id)
    if others.empty:
        return pd.Series(0.0, index=interaction_matrix.columns)
    target_norm = np.linalg.norm(target)
    sims = []
    for _, row in others.iterrows():
        v = row.values
        denom = target_norm * np.linalg.norm(v)
        sims.append(float(np.dot(target, v) / denom) if denom > 0 else 0.0)
    sims = np.array(sims)
    if sims.sum() <= 0:
        return pd.Series(0.0, index=interaction_matrix.columns)
    weighted = others.T.values @ sims
    weighted = weighted / (sims.sum() + 1e-9)
    return pd.Series(weighted, index=interaction_matrix.columns)


def collaborative_scores_for_couple(user_a_id, user_b_id, interaction_matrix, candidate_ids):
    collab_a = simulate_collaborative_filter(user_a_id, interaction_matrix)
    collab_b = simulate_collaborative_filter(user_b_id, interaction_matrix)
    combined = (collab_a + collab_b) / 2.0
    return combined.reindex(candidate_ids).fillna(0.0)


## [6. 🇧🇷Ranking Híbrido / 🇬🇧Hybrid Ranking]()
<br><br>

### [***🇧🇷Combinação ponderada de conteúdo e colaborativo / 🇬🇧Weighted combination of content and collaborative scores***]


<br>

🇧🇷
`content_score`, `collaborative_score` e `final_score` são sempre exibidos
separadamente, nunca apenas o número final.

#

🇬🇧
*`content_score`, `collaborative_score`, and `final_score` are always displayed separately, never just the final number.*

In [80]:
def _normalize(series):
    lo, hi = series.min(), series.max()
    if hi - lo < 1e-9:
        return pd.Series(0.5, index=series.index)
    return (series - lo) / (hi - lo)


def weighted_rank(candidates_df, content_scores, collaborative_scores,
                   weight_content=0.6, weight_collaborative=0.4):
    result = candidates_df.copy().reset_index(drop=True)
    content_scores = content_scores.reset_index(drop=True)
    collaborative_scores = collaborative_scores.reset_index(drop=True)

    result["content_score"] = content_scores.round(3)
    result["collaborative_score"] = collaborative_scores.round(3)
    final = (weight_content * _normalize(content_scores)
             + weight_collaborative * _normalize(collaborative_scores))
    result["final_score"] = final.round(3)
    return result.sort_values("final_score", ascending=False).reset_index(drop=True)


## [7. 🇧🇷Explicabilidade Estruturada e Bilíngue / 🇬🇧Structured and Bilingual Explainability]()
<br><br>

### [***🇧🇷Motivo da recomendação, em PT e EN / 🇬🇧Reason for recommendation, in PT and EN***]
<br>

In [81]:
def explain_recommendation(product_row, user_a, user_b):
    reasons_pt, reasons_en = [], []

    if (product_row["category"] in GOAL_CATEGORY.get(user_a["goals"], [])
            or product_row["category"] in GOAL_CATEGORY.get(user_b["goals"], [])):
        reasons_pt.append("alinhado aos objetivos do casal")
        reasons_en.append("matches the couple's goals")

    if user_a["dietary_restrictions"] != "none" or user_b["dietary_restrictions"] != "none":
        reasons_pt.append("respeita as restricoes alimentares informadas por ambos")
        reasons_en.append("respects both partners' dietary restrictions")

    if not reasons_pt:
        reasons_pt.append("boa avaliacao entre casais com perfil semelhante (simulado)")
        reasons_en.append("well rated among couples with a similar profile (simulated)")

    return {
        "reasons_pt": reasons_pt,
        "reasons_en": reasons_en,
        "text_pt": f"Recomendamos {product_row['description']} porque {', '.join(reasons_pt)}.",
        "text_en": f"We recommend {product_row['description']} because it is {', '.join(reasons_en)}.",
    }


## [8. 🇧🇷Saída de Voz (gTTS, opcional) / 🇬🇧Voice Output (gTTS, optional)]()
<br><br>

### [***🇧🇷Áudio não bloqueia a demonstração em caso de falha de rede / 🇬🇧Audio does not block the demo in case of network failure***]

<br>


🇧🇷
O `gTTS` depende de conexão com a internet. Qualquer falha é capturada e
registrada, e a execução continua normalmente, apenas sem áudio.

#  


🇬🇧
*`gTTS` relies on an internet connection. Any failure is caught and logged, and execution continues normally, just without audio.*

In [82]:
TTS_AVAILABLE = False
try:
    from gtts import gTTS
    TTS_AVAILABLE = True
except Exception:
    TTS_AVAILABLE = False


def speak(text, lang="pt"):
    if not TTS_AVAILABLE:
        print(f"[audio indisponivel - biblioteca gTTS ausente] {text}")
        return False
    try:
        tts_lang = "pt" if lang == "pt" else "en"
        tts = gTTS(text=text, lang=tts_lang)
        tts.save("habitsync_audio.mp3")
        print(f"[audio gerado com sucesso em habitsync_audio.mp3] ({tts_lang})")
        return True
    except Exception as e:
        print(f"[falha de rede/audio, continuando sem interromper a demo: {e}] {text}")
        return False


## [9. 🇧🇷Orquestração: Recomendação para um Casal / 🇬🇧Orchestration: Recommendation for a Couple]()
<br><br>

### [***🇧🇷Pipeline completo: seguro → conteúdo → colaborativo → ranking → explicação***]
### [***🇬🇧Full Pipeline: safe → content → collaborative → ranking → explanation***]
<br>

In [83]:
interaction_matrix = build_interaction_matrix(interactions_df, products_df)


def recommend_for_couple(couple_id, top_k=3, lang="pt", speak_result=False):
    couple_users = users_df[users_df["couple_id"] == couple_id]
    assert len(couple_users) == 2, f"couple_id {couple_id} nao possui exatamente 2 usuarios"
    user_a, user_b = couple_users.iloc[0], couple_users.iloc[1]

    # PRIORIDADE ABSOLUTA: filtro de seguranca antes de qualquer ranking
    safe_mask = products_df.apply(lambda p: is_safe_for_couple(p, user_a, user_b), axis=1)
    candidates_df = products_df[safe_mask & products_df["availability"]].reset_index(drop=True)

    content_scores = content_scores_for_couple(candidates_df, user_a, user_b)
    collaborative_scores = collaborative_scores_for_couple(
        user_a["user_id"], user_b["user_id"], interaction_matrix, candidates_df["product_id"]
    )
    ranked = weighted_rank(candidates_df, content_scores, collaborative_scores)
    top = ranked.head(top_k).copy()

    explanations = [explain_recommendation(row, user_a, user_b) for _, row in top.iterrows()]
    top["explanation_pt"] = [e["text_pt"] for e in explanations]
    top["explanation_en"] = [e["text_en"] for e in explanations]

    if speak_result and len(top) > 0:
        speak(top.iloc[0]["explanation_pt" if lang == "pt" else "explanation_en"], lang=lang)

    return top, user_a, user_b


## [10. 🇧🇷Agente Conversacional (entrada de texto simulada) / 🇬🇧Conversational Agent (simulated text input)]()
<br><br>

### [***🇧🇷Suporte a comandos em português e inglês / 🇬🇧Support for Portuguese and English commands***]

<br>


🇧🇷
A entrada permanece simulada por texto (`input()` substituído por strings
diretas nos cenários abaixo), como no `assistenteVoz.ipynb` original.

#

 🇬🇧
*Input remains simulated via text (`input()` replaced by direct strings in the scenarios below), just like the original `assistenteVoz.ipynb`.*

In [84]:
def detect_lang(text):
    text_l = text.lower()
    en_markers = ["recommend", "please", "suggest", "hello", "hi "]
    return "en" if any(m in text_l for m in en_markers) else "pt"


def parse_command(text):
    text_l = text.lower().strip()
    if any(w in text_l for w in ["recomend", "sugest", "recommend", "suggest"]):
        return {"intent": "recommend"}
    if any(w in text_l for w in ["ola", "oi", "hello", "hi"]):
        return {"intent": "greet"}
    if any(w in text_l for w in ["sair", "tchau", "exit", "bye"]):
        return {"intent": "exit"}
    return {"intent": "unknown"}


def build_response(couple_id, text, speak_result=False):
    lang = detect_lang(text)
    intent = parse_command(text)["intent"]

    if intent == "greet":
        response = t("greet", lang)
        if speak_result:
            speak(response, lang)
        return response, None

    if intent == "exit":
        response = t("exit", lang)
        if speak_result:
            speak(response, lang)
        return response, None

    if intent == "recommend":
        top, user_a, user_b = recommend_for_couple(couple_id, lang=lang, speak_result=speak_result)
        return f"OK ({lang})", top

    response = t("unknown", lang)
    if speak_result:
        speak(response, lang)
    return response, None


## [11. 🇧🇷Testes (assert-based) / 🇬🇧Tests (assert-based)]()
<br><br>

### [***🇧🇷Validação automática antes da demonstração / 🇬🇧Automatic validation before the demo***]
<br>

In [85]:
print("=== TESTES ===")

_top, _a, _b = recommend_for_couple("DEMO_COMPAT", lang="pt")
for col in ["content_score", "collaborative_score", "final_score"]:
    assert col in _top.columns, f"coluna ausente: {col}"
print("[OK] scores separados presentes (content/collaborative/final)")

_top_conflict, _a2, _b2 = recommend_for_couple("DEMO_CONFLICT", lang="pt")
assert "P902" not in _top_conflict["product_id"].values, "produto inseguro (nuts) foi recomendado!"
assert "P903" in _top_conflict["product_id"].values, "alternativa segura esperada nao apareceu no top"
print("[OK] prioridade de seguranca: produto com alergeno excluido, alternativa segura presente")

assert _top.iloc[0]["explanation_pt"] != _top.iloc[0]["explanation_en"]
assert len(_top.iloc[0]["explanation_pt"]) > 0 and len(_top.iloc[0]["explanation_en"]) > 0
print("[OK] explicacoes bilingues geradas corretamente")

assert detect_lang("Please recommend products for us") == "en"
assert detect_lang("Recomende produtos para nos") == "pt"
print("[OK] deteccao de idioma do comando")


=== TESTES ===
[OK] scores separados presentes (content/collaborative/final)
[OK] prioridade de seguranca: produto com alergeno excluido, alternativa segura presente
[OK] explicacoes bilingues geradas corretamente
[OK] deteccao de idioma do comando


## [12. 🇧🇷Cenários de Demonstração / 🇬🇧Demo Scenarios]()
<br><br>

### 🇧🇷 [***1. Comando em português***]()

<br>

In [86]:
resp, top1 = build_response("C001", "Recomende produtos para nos", speak_result=True)
print(resp)
top1[["product_id", "description", "content_score", "collaborative_score", "final_score", "explanation_pt"]]


[audio gerado com sucesso em habitsync_audio.mp3] (pt)
OK (pt)


,product_id,description,content_score,collaborative_score,final_score,explanation_pt
0,P017,Omega 3 product for vitamins goals,2.0,0.759,0.788,Recomendamos Omega 3 product for vitamins goal...
1,P009,Multivitamin product for vitamins goals,2.0,0.491,0.703,Recomendamos Multivitamin product for vitamins...
2,P021,Snack Bar product for healthy food goals,2.0,0.390,0.671,Recomendamos Snack Bar product for healthy foo...


### [***🇬🇧 2. Command in English***]()

In [87]:
resp2, top2 = build_response("C002", "Please recommend products for us", speak_result=True)
print(resp2)
top2[["product_id", "description", "content_score", "collaborative_score", "final_score", "explanation_en"]]


[audio gerado com sucesso em habitsync_audio.mp3] (en)
OK (en)


,product_id,description,content_score,collaborative_score,final_score,explanation_en
0,P005,Whey product for protein goals,2.5,1.280,1.000,We recommend Whey product for protein goals be...
1,P002,Vegan Protein product for protein goals,2.5,0.953,0.882,We recommend Vegan Protein product for protein...
2,P010,Creatine Monohydrate product for creatine goals,2.5,0.940,0.878,We recommend Creatine Monohydrate product for ...


### [***3. 🇧🇷Casal com objetivos compatíveis / 🇬🇧Couple with compatible goals***]()

<br>


In [88]:
top3, ua, ub = recommend_for_couple("DEMO_COMPAT", lang="pt", speak_result=True)
print(f"partner_a: goals={ua['goals']}, restriction={ua['dietary_restrictions']}")
print(f"partner_b: goals={ub['goals']}, restriction={ub['dietary_restrictions']}")
top3[["product_id", "description", "content_score", "collaborative_score", "final_score", "explanation_pt"]]


[audio gerado com sucesso em habitsync_audio.mp3] (pt)
partner_a: goals=muscle_gain, restriction=vegan
partner_b: goals=muscle_gain, restriction=vegetarian


,product_id,description,content_score,collaborative_score,final_score,explanation_pt
0,P901,Vegan Protein Blend for muscle gain goals,3.0,1.165,1.000,Recomendamos Vegan Protein Blend for muscle ga...
1,P002,Vegan Protein product for protein goals,3.0,1.038,0.938,Recomendamos Vegan Protein product for protein...
2,P020,Creatine Monohydrate product for creatine goals,2.5,0.344,0.502,Recomendamos Creatine Monohydrate product for ...


### [***4. Conflito de restrições — prioridade para a alternativa segura / Conflict of restrictions — priority to the safe alternative***]()

<br>

🇧🇷
`partner_b` possui `nut_allergy`. O produto P902 (contém `nuts`) tem alta
pontuação de conteúdo para o objetivo `muscle_gain`, mas é **excluído antes
do ranking** por violar a restrição de segurança — o sistema recomenda a
alternativa segura equivalente (P903).

---

🇬🇧
*`partner_b` has a `nut_allergy`. Product P902 (contains `nuts`) has a high content score for the `muscle_gain` goal, but is **excluded before ranking** because it violates the safety restriction — the system recommends the safe equivalent alternative (P903).*


In [89]:
top4, ua2, ub2 = recommend_for_couple("DEMO_CONFLICT", lang="pt", speak_result=True)
assert "P902" not in top4["product_id"].values, "falha de seguranca: produto com nuts foi recomendado"
print(f"partner_a: goals={ua2['goals']}, restriction={ua2['dietary_restrictions']}")
print(f"partner_b: goals={ub2['goals']}, restriction={ub2['dietary_restrictions']} (ALERGIA A CASTANHAS)")
top4[["product_id", "description", "allergens", "content_score", "collaborative_score", "final_score", "explanation_pt"]]


[audio gerado com sucesso em habitsync_audio.mp3] (pt)
partner_a: goals=muscle_gain, restriction=none
partner_b: goals=endurance, restriction=nut_allergy (ALERGIA A CASTANHAS)


,product_id,description,allergens,content_score,collaborative_score,final_score,explanation_pt
0,P005,Whey product for protein goals,lactose,1.5,1.168,1.000,Recomendamos Whey product for protein goals po...
1,P901,Vegan Protein Blend for muscle gain goals,none,1.5,1.129,0.985,Recomendamos Vegan Protein Blend for muscle ga...
2,P903,Nut-Free Whey Isolate for muscle gain goals,none,1.5,0.967,0.923,Recomendamos Nut-Free Whey Isolate for muscle ...


### 🇧🇷 [Experimente seu próprio comando aqui! / 🇬🇧 Try your own command here!]()Z



🇧🇷
Altere o texto da variável `user_input` abaixo para testar diferentes comandos e idiomas.

---

🇬🇧
Altere o texto da variável `user_input` abaixo para testar diferentes comandos e idiomas.

In [90]:
# Altere o texto abaixo para o seu comando (em português ou inglês)
user_input = "Recomende produtos para o casal C003"

# Escolha o casal para o qual deseja recomendações (ex: "C001", "C002", "DEMO_COMPAT", "DEMO_CONFLICT")
# O casal C003 é apenas um exemplo. Certifique-se de que o casal exista no dataframe 'users_df'.
couple_id_for_input = "C003" # Altere para o ID do casal desejado

# Executa o assistente com sua entrada
response, result_df = build_response(couple_id_for_input, user_input, speak_result=True)

print(response)
if result_df is not None:
    display(result_df[["product_id", "description", "content_score", "collaborative_score", "final_score", "explanation_pt", "explanation_en"]])

[audio gerado com sucesso em habitsync_audio.mp3] (pt)
OK (pt)


,product_id,description,content_score,collaborative_score,final_score,explanation_pt,explanation_en
0,P901,Vegan Protein Blend for muscle gain goals,2.0,1.007,1.000,Recomendamos Vegan Protein Blend for muscle ga...,We recommend Vegan Protein Blend for muscle ga...
1,P002,Vegan Protein product for protein goals,2.0,0.954,0.978,Recomendamos Vegan Protein product for protein...,We recommend Vegan Protein product for protein...
2,P903,Nut-Free Whey Isolate for muscle gain goals,2.0,0.889,0.950,Recomendamos Nut-Free Whey Isolate for muscle ...,We recommend Nut-Free Whey Isolate for muscle ...


### 🇧🇷Interface de Conversação com ipywidgets / 🇬🇧Conversational Interface with ipywidgets

In [91]:
import ipywidgets as widgets
from IPython.display import display, HTML, Audio, Image # Import Image

# Pre-defined commands for the dropdown - now with English first
PREDEFINED_COMMANDS = [
    "Select a command...", # Placeholder (English)
    "Recommend products for us",
    "Suggest products for us",
    "Hi",
    "Bye",
    "Selecione um comando...", # Placeholder (Portuguese)
    "Recomende produtos para nós",
    "Sugira produtos para nós",
    "Olá",
    "Sair"
]

# Custom names for couples
CUSTOM_COUPLE_NAMES = {
    "C001": "Eros e Psiquê",
    "C002": "Adão e Eva",
    "C003": "Rama e Sita",
    "C004": "Zeus e Hera",
    "C005": "Hades e Perséfone",
    "C006": "Ulisses e Penélope",
    "C007": "Orfeu e Eurídice",
    "C008": "Teseu e Ariadne",
    "C009": "Píramo e Tisbe",
    "C010": "Ísis e Osíris",
    "DEMO_COMPAT": "Romeu e Julieta",
    "DEMO_CONFLICT": "Bonnie e Clyde" # Adding one more for consistency
}

# Dictionary to map couple_id to a visual (image path)
COUPLE_VISUALS = {
    "C001": "/content/Screenshot 2026-09-02 at 03.54.25.png", # Example image 1
    "C002": "/content/Screenshot 2026-09-02 at 04.04.55.png", # Example image 2
    "C003": "/content/Screenshot 2026-09-02 at 03.54.25.png",
    "C004": "/content/Screenshot 2026-09-02 at 04.04.55.png",
    "C005": "/content/Screenshot 2026-09-02 at 03.54.25.png",
    "C006": "/content/Screenshot 2026-09-02 at 04.04.55.png",
    "C007": "/content/Screenshot 2026-09-02 at 03.54.25.png",
    "C008": "/content/Screenshot 2026-09-02 at 04.04.55.png",
    "C009": "/content/Screenshot 2026-09-02 at 04.04.55.png",
    "C010": "/content/Screenshot 2026-09-02 at 04.04.55.png",
    "DEMO_COMPAT": "/content/Screenshot 2026-09-02 at 03.54.25.png",
    "DEMO_CONFLICT": "/content/Screenshot 2026-09-02 at 04.04.55.png",
}

# Common layout for input widgets
common_input_field_layout = widgets.Layout(width='300px', border='2px solid magenta', margin='5px')
common_tab_layout = widgets.Layout(width='310px', border='2px solid magenta', margin='5px') # Slightly wider to contain children
common_button_layout = widgets.Layout(width='940px', border='2px solid magenta', margin='5px') # Adjusted button width to match combined inputs (310+310+320 = 940)

# Widget de entrada para o comando do usuário - Dropdown (comandos predefinidos)
command_dropdown = widgets.Dropdown(
    options=PREDEFINED_COMMANDS,
    value=PREDEFINED_COMMANDS[0], # Default to English placeholder
    description=t("command_dropdown_desc", "en"), # Default description in English
    disabled=False,
    layout=common_input_field_layout
)

# Widget de entrada para o comando do usuário - Text Input (comandos personalizados)
custom_command_widget = widgets.Text(
    value='',
    placeholder=t("custom_command_placeholder", "en"), # Default placeholder in English
    description=t("custom_command_desc", "en"), # Default description in English
    disabled=False,
    layout=common_input_field_layout
)

# Criar as abas para os comandos
command_tabs = widgets.Tab()
command_tabs.children = [command_dropdown, custom_command_widget]
command_tabs.set_title(0, t("tab_suggested_commands_title", "en")) # Default title in English
command_tabs.set_title(1, t("tab_custom_command_title", "en")) # Default title in English
command_tabs.layout = common_tab_layout

# Widget de seleção de idioma - English first, default to English
language_dropdown = widgets.Dropdown(
    options=[('🇬🇧English', 'en'), ('🇧🇷Português', 'pt')], # English first, with flags
    value='en', # Default to English
    description=t("language_dropdown_desc", "en"), # Default description in English
    disabled=False,
    layout=common_input_field_layout
)

# Prepare initial couple options list
initial_couple_options = []

# Add a placeholder option at the beginning
# This will now use 'en' as the default language from language_dropdown.value
initial_couple_options.append((t("select_couple_placeholder", language_dropdown.value), ""))

for couple_id in users_df['couple_id'].unique():
    couple_users = users_df[users_df['couple_id'] == couple_id]
    if len(couple_users) == 2:
        user_a = couple_users[couple_users['role'] == 'partner_a'].iloc[0]
        user_b = couple_users[couple_users['role'] == 'partner_b'].iloc[0]

        # Get custom name if available, otherwise use couple_id
        couple_name = CUSTOM_COUPLE_NAMES.get(couple_id, couple_id)

        # Initial display name in English
        display_name = (f"{couple_name} ({couple_id}) ("f"{t('partner_a_label', 'en')}{user_a['goals'].replace('_', ' ')}, "f"{t('partner_b_label', 'en')}{user_b['goals'].replace('_', ' ')})")
        initial_couple_options.append((display_name, couple_id))
    else:
        # Default message in English
        initial_couple_options.append((f"{couple_id}{t('couple_incomplete_info', 'en')}", couple_id))

# Widget de entrada para o ID do casal (dropdown)
couple_id_widget = widgets.Dropdown(
    options=initial_couple_options,
    value="", # Set default value to the placeholder's value
    description=t("couple_id_desc", "en"), # Default description in English
    disabled=False,
    layout=common_input_field_layout
)

# Botão para enviar o comando
submit_button = widgets.Button(
    description=t("submit_button_desc", "en"), # Default description in English
    disabled=False,
    button_style='info',
    tooltip=t("submit_button_tooltip", "en"), # Default tooltip in English
    icon='send',
    layout=common_button_layout
)

# Área de saída para exibir os resultados
output_area = widgets.Output(layout=widgets.Layout(border='1px solid lightgray', padding='10px', margin='5px'))

def update_widget_language(lang):
    # Update descriptions and placeholders
    command_dropdown.description = t("command_dropdown_desc", lang)
    custom_command_widget.description = t("custom_command_desc", lang)
    custom_command_widget.placeholder = t("custom_command_placeholder", lang)
    command_tabs.set_title(0, t("tab_suggested_commands_title", lang))
    command_tabs.set_title(1, t("tab_custom_command_title", lang))
    couple_id_widget.description = t("couple_id_desc", lang)
    language_dropdown.description = t("language_dropdown_desc", lang)
    submit_button.description = t("submit_button_desc", lang)
    submit_button.tooltip = t("submit_button_tooltip", lang)

    # Rebuild couple_options to update the display names in the correct language
    new_couple_options = []
    new_couple_options.append((t("select_couple_placeholder", lang), "")) # Placeholder

    for couple_id in users_df['couple_id'].unique():
        couple_users = users_df[users_df['couple_id'] == couple_id]
        if len(couple_users) == 2:
            user_a = couple_users[couple_users['role'] == 'partner_a'].iloc[0]
            user_b = couple_users[couple_users['role'] == 'partner_b'].iloc[0]

            couple_name = CUSTOM_COUPLE_NAMES.get(couple_id, couple_id)

            goal_a_text = user_a['goals'].replace('_', ' ')
            goal_b_text = user_b['goals'].replace('_', ' ')

            display_name = (f"{couple_name} ({couple_id}) ("f"{t('partner_a_label', lang)}{goal_a_text}, "f"{t('partner_b_label', lang)}{goal_b_text})")
            new_couple_options.append((display_name, couple_id))
        else:
            new_couple_options.append((f"{couple_id}{t('couple_incomplete_info', lang)}", couple_id))

    # Preserve the current value if it's still available in the new options
    current_couple_value = couple_id_widget.value
    couple_id_widget.options = new_couple_options
    if current_couple_value in [opt[1] for opt in new_couple_options]:
        couple_id_widget.value = current_couple_value
    else:
        couple_id_widget.value = "" # Reset to placeholder if current value is invalid


def on_button_click(b):
    with output_area:
        output_area.clear_output()

        selected_lang = language_dropdown.value

        user_command = ""
        # Determine the user command based on the active tab
        if command_tabs.selected_index == 0: # Suggested Commands tab
            # Check if the selected command is one of the placeholder values
            if command_dropdown.value != PREDEFINED_COMMANDS[0] and command_dropdown.value != PREDEFINED_COMMANDS[5]:
                user_command = command_dropdown.value
        else: # Custom Command tab
            user_command = custom_command_widget.value

        couple_id = couple_id_widget.value

        if not user_command or not couple_id:
            print(t("error_empty_input", selected_lang))
            return

        # Get the custom couple name for styling
        couple_name_for_display = CUSTOM_COUPLE_NAMES.get(couple_id, couple_id)

        # Apply magenta and cyan decoration with Kavanna font to the processing message
        styled_user_command = f"<span style='color: cyan; font-family: 'Kavanna', sans-serif;'>'{user_command}'</span>"
        styled_couple_name = f"<span style='color: magenta; font-family: 'Kavanna', sans-serif;'>'{couple_name_for_display}'</span>"

        processing_message_template = t("processing_command", selected_lang)
        processing_message = processing_message_template.format(
            user_command=styled_user_command,
            couple_id=styled_couple_name,
            selected_lang=selected_lang
        )
        display(HTML(processing_message))

        # Call build_response which already integrates speech and audio playback
        response_text, result_df = build_response(couple_id, user_command, speak_result=True)

        print(response_text)

        # Display couple-specific image if available
        if couple_id in COUPLE_VISUALS:
            image_path = COUPLE_VISUALS[couple_id]
            try:
                display(Image(filename=image_path, width=400))
            except FileNotFoundError:
                print(f"Aviso: Arquivo de imagem não encontrado para o casal {couple_id}: {image_path}")
            except Exception as e:
                print(f"Erro ao exibir imagem para o casal {couple_id}: {e}")

        if result_df is not None:
            explanation_col = "explanation_pt" if selected_lang == "pt" else "explanation_en"
            display(result_df[["product_id", "description", "content_score", "collaborative_score", "final_score", explanation_col]])

        # The audio is saved as 'habitsync_audio.mp3' by the speak() function.
        # Here we simply load and display it.
        try:
            display(Audio('habitsync_audio.mp3', autoplay=True))
        except Exception as e:
            print(t("audio_error", selected_lang).format(e=e))

# Connect the button to the click function
submit_button.on_click(on_button_click)

# Attach observer to language_dropdown for real-time updates
language_dropdown.observe(lambda change: update_widget_language(change.new), names='value')

# Initial language setup based on default language_dropdown value
update_widget_language(language_dropdown.value)

# Create layout for input controls
input_controls = widgets.VBox([
    widgets.HBox([
        command_tabs,
        couple_id_widget,
        language_dropdown
    ], layout=widgets.Layout(justify_content='center')),
    submit_button
], layout=widgets.Layout(align_items='center'))

# Display the widgets in a more organized layout
display(input_controls, output_area)

Output(layout=Layout(border='1px solid lightgray', margin='5px', padding='10px'))

### 🇧🇷Recomendação para outro casal (Zeus e Hera - C004) / 🇬🇧Recommendation for another couple (Zeus and Hera - C004)

In [92]:
couple_id_new = "C004"
user_input_new = "Recomende produtos para nós"

print(f"Executando recomendação para o casal {couple_id_new}:")
resp_new, result_df_new = build_response(couple_id_new, user_input_new, speak_result=True)

print(resp_new)
display(result_df_new[["product_id", "description", "content_score", "collaborative_score", "final_score", "explanation_pt", "explanation_en"]])

Executando recomendação para o casal C004:
[audio gerado com sucesso em habitsync_audio.mp3] (pt)
OK (pt)


,product_id,description,content_score,collaborative_score,final_score,explanation_pt,explanation_en
0,P009,Multivitamin product for vitamins goals,2.0,1.208,0.935,Recomendamos Multivitamin product for vitamins...,We recommend Multivitamin product for vitamins...
1,P025,Vitamin D product for vitamins goals,2.0,0.495,0.678,Recomendamos Vitamin D product for vitamins go...,We recommend Vitamin D product for vitamins go...
2,P021,Snack Bar product for healthy food goals,2.0,0.437,0.657,Recomendamos Snack Bar product for healthy foo...,We recommend Snack Bar product for healthy foo...


### 🇧🇷 [Nota sobre o áudio]()

Se você não estiver ouvindo o áudio automaticamente, é provável que seja uma configuração do seu navegador ou um problema intermitente com o player de áudio do Colab. O arquivo `habitsync_audio.mp3` está sendo gerado corretamente a cada interação, conforme indicado pela mensagem `[audio gerado com sucesso...]`.

Você pode tentar as seguintes soluções:
1.  Verificar as configurações de áudio do seu navegador.
2.  Atualizar a página do Colab.
3.  Baixar o arquivo de áudio diretamente através do link abaixo para reprodução externa.

---

### 🇬🇧 [Note on audio]()

*If you are not hearing audio automatically, it is likely a browser setting or an intermittent issue with the Colab audio player. The `habitsync_audio.mp3` file is being generated correctly with each interaction, as indicated by the `[audio gerado com sucesso...]` message.*

*You can try the following solutions:*
*1. Check your browser's audio settings.*
*2. Refresh the Colab page.*
*3. Download the audio file directly via the link below for external playback.*

In [93]:
from IPython.display import FileLink

# Fornece um link para baixar o arquivo de áudio gerado
# Provides a link to download the generated audio file
display(FileLink('habitsync_audio.mp3'))

/content/habitsync_audio.mp3

### 🇧🇷Demonstração atualizada: Cenário 1 (Português) com áudio integrado / 🇬🇧Updated Demo: Scenario 1 (Portuguese) with integrated audio

In [94]:
print('Executando Cenário 1 (Português) com áudio:')
resp, top1 = build_response("C001", "Recomende produtos para nos", speak_result=True)
print(resp)
display(top1[["product_id", "description", "content_score", "collaborative_score", "final_score", "explanation_pt"]])

Executando Cenário 1 (Português) com áudio:
[audio gerado com sucesso em habitsync_audio.mp3] (pt)
OK (pt)


,product_id,description,content_score,collaborative_score,final_score,explanation_pt
0,P017,Omega 3 product for vitamins goals,2.0,0.759,0.788,Recomendamos Omega 3 product for vitamins goal...
1,P009,Multivitamin product for vitamins goals,2.0,0.491,0.703,Recomendamos Multivitamin product for vitamins...
2,P021,Snack Bar product for healthy food goals,2.0,0.390,0.671,Recomendamos Snack Bar product for healthy foo...


### 🇧🇷Demonstração atualizada: Cenário 2 (Inglês) com áudio integrado / 🇬🇧Updated Demo: Scenario 2 (English) with integrated audio

In [95]:
print('Executing Scenario 2 (English) with audio:')
resp2, top2 = build_response("C002", "Please recommend products for us", speak_result=True)
print(resp2)
display(top2[["product_id", "description", "content_score", "collaborative_score", "final_score", "explanation_en"]])

Executing Scenario 2 (English) with audio:
[audio gerado com sucesso em habitsync_audio.mp3] (en)
OK (en)


,product_id,description,content_score,collaborative_score,final_score,explanation_en
0,P005,Whey product for protein goals,2.5,1.280,1.000,We recommend Whey product for protein goals be...
1,P002,Vegan Protein product for protein goals,2.5,0.953,0.882,We recommend Vegan Protein product for protein...
2,P010,Creatine Monohydrate product for creatine goals,2.5,0.940,0.878,We recommend Creatine Monohydrate product for ...


In [96]:
from IPython.display import Audio

# Display the generated audio file in the notebook
Audio('habitsync_audio.mp3')

In [97]:
print(t("banner_synthetic", "pt"))
print(t("banner_synthetic", "en"))
print()
print("NB COMPLETED SUCCESSFULLY — READY FOR next NB")


Aviso: todos os dados usados sao sinteticos (fins educacionais). As recomendacoes sao informativas e nao substituem orientacao nutricional, medica ou de treinamento profissional.
Notice: all data used is synthetic (educational purposes only). Recommendations are informational and do not replace professional nutrition, medical, or training guidance.

NB COMPLETED SUCCESSFULLY — READY FOR next NB
